# LFW 02. Compressor fit

> **Step 1 압축군**: 원본 512D에서 PCA-384/256/128/64/32를 각각 독립 학습하고, PQ도 PCA 출력이 아닌 동일한 원본 512D에서 직접 학습합니다. 예상 시간은 현재 CPU에서 10~40분이며 중단되면 처음부터 새 attempt로 실행합니다.

## 예상 소요 시간 (현재 LFW 13,195개 임베딩·CPU 기준)

| 실행 모드 | 예상 시간 | 주로 오래 걸리는 구간 |
| --- | ---: | --- |
| `EXECUTE_STAGE=False` | 1초 미만 | run 연결과 입력 존재 여부만 확인 |
| `EXECUTE_STAGE=True` | 약 3~15분 | DB 벡터 로딩, PCA 학습, Faiss PQ CPU 학습 |

> 실제 실행 중에는 30초마다 `RUNNING` heartbeat와 누적 시간이 출력됩니다. heartbeat가 계속 나오면 멈춘 것이 아닙니다.

목표: development identity의 원본 임베딩만 사용해 PCA와 Faiss PQ를 학습하고, 모델과 학습 요약을 run artifact로 고정합니다. 성공 기준은 test/calibration 누수 없이 모델 파일의 hash와 학습 표본 수가 기록되는 것입니다.

> **재시작/재개 규칙(필수)**: 임의 셀에서 시작하지 말고 **Kernel Restart 후 Run All**을 사용합니다. 01이 완료되고 원본 임베딩 수가 확인된 run에서만 실행합니다. 중단되면 02 전체를 다시 실행해 새 attempt를 만들며, attempt 번호가 붙은 기존 모델 artifact는 덮어쓰지 않습니다. manifest/development split이 바뀌면 00부터 새 run을 시작합니다.


In [1]:
# Step 1 실행 범위: 이 셀의 세 값만 바꾸고 Kernel Restart -> Run All
MODE = 'real'             # 'dev' 또는 'real'
DATA_FRACTION = 1.0     # 0 < DATA_FRACTION <= 1
SEED = 42

import sys
from pathlib import Path

for _scope_root in (Path.cwd(), *Path.cwd().parents):
    if (_scope_root / 'research').is_dir():
        break
else:
    raise FileNotFoundError('D:/ronbun 내부에서 노트북을 실행하십시오.')
if str(_scope_root) not in sys.path:
    sys.path.insert(0, str(_scope_root))

from research.compression import PCA_SWEEP_DIMENSIONS
from research.experiments.scope import ExperimentScope

PCA_DIMENSIONS = (384, 256, 128, 64, 32)
PQ_SOURCE_DIMENSION = 512
if PCA_DIMENSIONS != tuple(PCA_SWEEP_DIMENSIONS):
    raise RuntimeError('노트북 PCA sweep과 공통 압축 정의가 다릅니다.')
EXPERIMENT_SCOPE = ExperimentScope(
    mode=MODE, data_fraction=DATA_FRACTION, seed=SEED
)
EXPERIMENT_SCOPE.as_dict()


{'mode': 'real',
 'data_fraction': 1.0,
 'seed': 42,
 'is_full_dataset': True,
 'is_paper_run': True}

In [2]:
from __future__ import annotations

import json
import os
from pathlib import Path

def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'research').is_dir() and (candidate / 'configs').is_dir():
            return candidate.resolve()
    raise RuntimeError('Run Jupyter from the ronbun repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
from research.runtime import ProgressReporter, RunStore, resolve_active_run

EXECUTE_STAGE = True
RUN_ROOT = PROJECT_ROOT / 'runs' / 'lfw'
LEGACY_RUN_ROOT = PROJECT_ROOT / 'runs'
try:
    RUN_DIR = resolve_active_run(RUN_ROOT)
except FileNotFoundError:
    RUN_ROOT = LEGACY_RUN_ROOT
    RUN_DIR = resolve_active_run(RUN_ROOT)
PROGRESS = ProgressReporter('02 compressor fit', heartbeat_seconds=30)


## Plan

- Join DB embeddings to manifest paths and retain only `split == development`.
- Fit every PCA dimension independently from origin-512.
- Fit PQ directly from origin-512; never feed a PCA projection to PQ.
- Save attempt-specific models and record model/profile metadata.


In [3]:
def attach_run(run_dir: Path) -> tuple[RunStore, dict]:
    run = RunStore.open(run_dir)
    manifest = json.loads(run.manifest_path.read_text(encoding='utf-8'))
    if manifest.get('status') == 'completed' or (run_dir / 'COMPLETED').exists():
        raise RuntimeError('Completed runs are immutable.')
    return run, manifest

preflight = {
    'execute_stage': EXECUTE_STAGE,
    'run_dir_resolved': str(RUN_DIR),
    'run_manifest_exists': bool(RUN_DIR and (RUN_DIR / 'run_manifest.json').is_file()),
}
preflight


{'execute_stage': True,
 'run_dir_resolved': 'C:\\ronbun\\runs\\lfw\\2026\\07\\27\\20260727-R002-9bf758ff_thesis3_lfw_face_search_v1',
 'run_manifest_exists': True}

## Execute and record

PCA/PQ 학습 표본은 반드시 development split으로 제한합니다. 두 family는 같은 원본 512D 입력에서 갈라지는 독립 실험군입니다. PQ code는 차원이 아니라 `m × nbits`로 크기를 기록합니다.


In [4]:
result = {'status': 'not_executed', **preflight}
if EXECUTE_STAGE:
    import numpy as np
    import pandas as pd
    from research.compression import ORIGIN_512, PCACompressor, PQCompressor
    from research.database import create_database_engine, load_database_settings, session_scope
    from research.database.models import Embedding512, Image

    with PROGRESS.step('run/input 및 01 artifact 검증', expected='10초 미만'):
        run, run_manifest = attach_run(RUN_DIR)
        run.verify_inputs()
        run.verify_phase_artifacts('01_arcface_embedding_extraction')
        config = run_manifest['config']
        manifest = pd.read_csv(PROJECT_ROOT / config['dataset']['manifest_path'])
        development_paths = {
            str((PROJECT_ROOT / Path(str(path))).resolve())
            if not Path(str(path)).is_absolute()
            else str(Path(str(path)).resolve())
            for path in manifest.loc[manifest['split'].eq('development'), 'image_path']
        }

    with PROGRESS.step('development 원본 벡터 DB 로딩', expected='10초~2분'):
        engine = create_database_engine(load_database_settings())
        with session_scope(engine) as session:
            db_rows = (
                session.query(Embedding512, Image)
                .join(Image, Embedding512.image_id == Image.id)
                .filter(
                    Embedding512.vector_type == ORIGIN_512,
                    Embedding512.run_uid == run.run_id,
                )
                .all()
            )
            vectors = [
                np.asarray(embedding.embedding, dtype=np.float32)
                for embedding, image in db_rows
                if str(Path(image.image_path).resolve()) in development_paths
            ]
        if not vectors:
            raise ValueError('No development ArcFace embeddings were found for this run.')
        matrix = np.stack(vectors)
        PROGRESS.emit('development 벡터 준비 완료', rows=len(matrix), dimensions=matrix.shape[1])

    pca_cfg = config['compression'].get('pca', {})
    pca_dimensions = [int(value) for value in pca_cfg.get('dimensions', list(PCA_DIMENSIONS))]
    if tuple(pca_dimensions) != PCA_DIMENSIONS:
        raise ValueError(f'LFW PCA sweep dimensions must be {PCA_DIMENSIONS}.')
    pq_cfg = config['compression'].get('pq', {})
    with run.phase('02_compressor_fit') as phase:
        suffix = f'A{phase.attempt:03d}'
        with PROGRESS.step('PCA 256D 학습', expected='1~10분'):
            pcas = {
                f'pca_{dimension}': PCACompressor(
                    dimension,
                    random_state=int(config['protocol'].get('split_seed', 42)),
                ).fit(matrix)
                for dimension in pca_dimensions
            }
        with PROGRESS.step('Faiss PQ 보조 codebook 학습', expected='5~30분'):
            pq = PQCompressor(
                matrix.shape[1],
                m=int(pq_cfg.get('m', 16)),
                nbits=int(pq_cfg.get('nbits', 8)),
            ).fit(matrix)
        with PROGRESS.step('compressor artifact 저장 및 hash 고정', expected='1분 미만'):
            pca_sources = {
                profile: compressor.save(phase.attempt_dir / f'{profile}_{suffix}.joblib')
                for profile, compressor in pcas.items()
            }
            pq_source = pq.save(phase.attempt_dir / f'pq_{suffix}.faiss')
            pca_artifacts = {
                profile: phase.publish_artifact(path)
                for profile, path in pca_sources.items()
            }
            pq_artifact = phase.publish_artifact(pq_source)
            summary = {
                'fit_split': 'development',
                'fit_count': int(len(matrix)),
                'source_dim': int(matrix.shape[1]),
                'pca_profiles': {
                    profile: {
                        'artifact': str(pca_artifacts[profile].relative_to(run.run_dir)),
                        'n_components': compressor.n_components,
                        'explained_variance_ratio_sum': float(
                            compressor.model.explained_variance_ratio_.sum()
                        ),
                    }
                    for profile, compressor in pcas.items()
                },
                'pq': {
                    'artifact': str(pq_artifact.relative_to(run.run_dir)),
                    'm': pq.m,
                    'nbits': pq.nbits,
                    'pgvector_searchable': False,
                },
            }
            summary_source = phase.attempt_dir / f'compressor_summary_{suffix}.json'
            summary_source.write_text(
                json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
            )
            phase.publish_artifact(summary_source)
            phase.record_counts(development_vectors=len(matrix))
    PROGRESS.emit('02 완료', run_id=run.run_id, fit_count=len(matrix))
    result = {'status': 'completed', 'run_id': run.run_id, **summary}
else:
    PROGRESS.emit('검토 모드 완료: DB 로딩과 PCA/PQ 학습을 실행하지 않음', expected='1초 미만')
result


[01:29:31] 02 compressor fit | START run/input 및 01 artifact 검증 | elapsed=0s | expected=10초 미만
[01:29:33] 02 compressor fit | DONE run/input 및 01 artifact 검증 | elapsed=2s | step_elapsed=2s
[01:29:33] 02 compressor fit | START development 원본 벡터 DB 로딩 | elapsed=2s | expected=10초~2분
[01:29:40] 02 compressor fit | development 벡터 준비 완료 | elapsed=9s | rows=7742 dimensions=512
[01:29:40] 02 compressor fit | DONE development 원본 벡터 DB 로딩 | elapsed=9s | step_elapsed=6s
[01:29:40] 02 compressor fit | START PCA 256D 학습 | elapsed=9s | expected=1~10분
[01:29:40] 02 compressor fit | DONE PCA 256D 학습 | elapsed=9s | step_elapsed=0s
[01:29:40] 02 compressor fit | START Faiss PQ 보조 codebook 학습 | elapsed=9s | expected=5~30분
[01:29:43] 02 compressor fit | DONE Faiss PQ 보조 codebook 학습 | elapsed=12s | step_elapsed=2s
[01:29:43] 02 compressor fit | START compressor artifact 저장 및 hash 고정 | elapsed=12s | expected=1분 미만
[01:29:43] 02 compressor fit | DONE compressor artifact 저장 및 hash 고정 | elapsed=12s | step_elap

{'status': 'completed',
 'run_id': '20260727-R002-9bf758ff',
 'fit_split': 'development',
 'fit_count': 7742,
 'source_dim': 512,
 'pca_profiles': {'pca_384': {'artifact': 'artifacts\\02_compressor_fit\\pca_384_A001.joblib',
   'n_components': 384,
   'explained_variance_ratio_sum': 0.9942256808280945},
  'pca_256': {'artifact': 'artifacts\\02_compressor_fit\\pca_256_A001.joblib',
   'n_components': 256,
   'explained_variance_ratio_sum': 0.8719143867492676},
  'pca_128': {'artifact': 'artifacts\\02_compressor_fit\\pca_128_A001.joblib',
   'n_components': 128,
   'explained_variance_ratio_sum': 0.6188725233078003},
  'pca_64': {'artifact': 'artifacts\\02_compressor_fit\\pca_64_A001.joblib',
   'n_components': 64,
   'explained_variance_ratio_sum': 0.40966928005218506},
  'pca_32': {'artifact': 'artifacts\\02_compressor_fit\\pca_32_A001.joblib',
   'n_components': 32,
   'explained_variance_ratio_sum': 0.2622566223144531}},
 'pq': {'artifact': 'artifacts\\02_compressor_fit\\pq_A001.fais

## Next step

fit count, PCA 차원, PQ 파라미터를 확인합니다. PQ 학습 실패 시 표본 수와 `m/nbits`를 기록한 뒤 설정 변경은 00의 새 run으로 수행합니다.
